# Inspect the spaces and maps of an encoding

How does a picture of regions tell us what a module does? We will construct the closed square from [finite encodings](../finite_encodings.md), follow two original parameters into the returned finite poset, and inspect the linear map between their spaces. The final figure combines the parameter plane, the finite order, and a matrix. A two-square variation shows why the matrix adds information that dimensions miss. We then inspect its active presentation blocks and embedded image bases to explain where those spaces and maps came from.

You need coordinatewise order on pairs and matrix multiplication. The [persistence modules](../persistence_modules.md) and [indicator presentations](../indicator_presentations.md) chapters explain those mathematical ingredients. No input files are needed.

Use a **Julia 1.12 or compatible newer kernel** in an environment with TamerOp and CairoMakie installed; IJulia supplies the Julia kernel for Jupyter. Follow the [package installation instructions](../../README.md) and [optional integrations guide](../optional_integrations.md) before running the cells. This notebook does not install packages or change your environment.

Our route is: construct the square, read the actual finite poset, inspect a map, try a pair with no map, then compare two overlapping squares, recover their spaces and maps from the presentation, and save the figures.

In [ ]:
import TamerOp as OP
import TamerOp.Advanced as OA
import TamerOp.CoreModules: QQField
using CairoMakie

field = QQField()
options = OA.EncodingOptions(; backend=:pl_backend,
    poset_kind=:signature, field=field)

## First calculate the square by hand

Let $U$ be the region above $(0,0)$ and $D$ the region below $(2,2)$. The image of the scalar-one map between their indicator modules is one-dimensional exactly on $U\cap D=[0,2]^2$. We use rational coefficients, denoted `QQ`, and include all four edges of the square.

At an interior point the space is $\mathbb{Q}$; outside the square it is zero. For two comparable interior points, the structure map is the $1\times1$ matrix $[1]$. A map leaving the square is the unique zero map into the zero space, represented by a $0\times1$ matrix.

`BoxUpset` and `BoxDownset` describe the two unbounded input regions. We import `Advanced` for these constructors and for detailed inspection. The canonical `OP.encode` call constructs an `EncodingResult`: its poset, its assignment of parameters to labels, and its encoded module belong together. Every corner used below is exactly representable by the box encoder.

In [ ]:
births = [OA.BoxUpset([0.0, 0.0])]
deaths = [OA.BoxDownset([2.0, 2.0])]
coefficient = reshape(OP.QQ[1], 1, 1)
enc = OP.encode(births, deaths, coefficient, options)

P = OP.encoding_poset(enc)
classifier = OP.encoding_map(enc)
dims = OP.dimensions(enc)
@assert OP.provenance(enc).field == field
(; nvertices=OA.nvertices(P), dimensions=dims)

## Read the returned finite object

The introductory nine-region grid is one valid encoding. This signature encoder groups points by two bits: membership in the birth upset and membership in the **complement** of the death downset. For this input the four signatures form a diamond. Numeric IDs come from the returned object; we discover them with `locate` instead of guessing their order.

`:hasse` draws only the cover relations: a solid arrow from one label to another has no intermediate label. Upward means increasing in the finite order. Positions in this diagram are schematic; they are not coordinates in the parameter plane. A module input adds each stalk's dimension. This view needs no structure-map query.

`OA.visual_spec` builds a renderer-independent description. `OP.visualize` renders it through the CairoMakie package loaded above. The same specification can later be saved.

In [ ]:
inside_id = OA.locate(classifier, (1//1, 1//1))
outside_id = OA.locate(classifier, (3//1, 1//1))
@assert dims[inside_id] == 1
@assert dims[outside_id] == 0
@assert dims[OA.locate(classifier, (2//1, 2//1))] == 1

order_spec = OA.visual_spec(enc; kind=:hasse, vertex=inside_id)
display(OP.visualize(order_spec; backend=:cairomakie))
(; inside_id, outside_id)

## Follow a pair of parameters into a matrix

Take $x=(1/2,1/2)$ and $y=(3/2,3/2)$. Both are in the square and $x\leq y$, so our hand calculation predicts $[1]$. Use `parameter_pair=(x,y)` to ask about these **original parameters**.

In the figure, colors and IDs agree between the parameter plane and finite poset. The space at the source supplies the **columns** of the matrix; the target supplies its **rows**. Rank counts independent vectors that survive, while kernel dimension counts independent vectors sent to zero. Here the answers are one and zero.

The readout uses the module's stored coordinate bases. It does not claim that the basis vectors are original cycles or specified columns in the input presentation. `inspection.matrix` retains the actual coefficient matrix, not numbers recovered from an image.

In [ ]:
x, y = (1//2, 1//2), (3//2, 3//2)
square_spec = OA.visual_spec(enc; kind=:module_inspector,
    parameter_pair=(x, y), box=([-1, -1], [3, 3]))
square_map = OA.visual_metadata(square_spec).inspection
@assert square_map.matrix == reshape(OP.QQ[1], 1, 1)
@assert (square_map.rank, square_map.kernel_dimension) == (1, 0)
display(OP.visualize(square_spec; backend=:cairomakie))

## Exercise: zero map or no map?

Predict two answers before running the next cell. First, move the target to $(3,1)$ while leaving the source at $(1/2,1/2)$. Second, choose $(1/4,3/2)$ and $(3/2,1/4)$, which lie inside the square and share its label.

The first pair is ordered and leaves the support, so its matrix has **zero rows and one column**. The second pair is incomparable: one coordinate increases while the other decreases. There is **no ambient structure map**, even though both points have the same finite label. Passing `pair=(inside_id,inside_id)` would instead ask for the identity in the finite model. The two requests express different mathematical questions.

An exterior point of this square still has a represented label with dimension zero. For other encodings, classifier label `0` can mean unrepresented; the inspector distinguishes it from a represented zero space.

In [ ]:
leaving = OA.visual_spec(enc; kind=:module_inspector,
    parameter_pair=(x, (3//1, 1//1)), box=([-1, -1], [4, 4]))
leaving_map = OA.visual_metadata(leaving).inspection
@assert leaving_map.defined && size(leaving_map.matrix) == (0, 1)

a, b = (1//4, 3//2), (3//2, 1//4)
@assert OA.locate(classifier, a) == OA.locate(classifier, b)
unordered = OA.visual_spec(enc; kind=:module_inspector, parameter_pair=(a, b))
unordered_map = OA.visual_metadata(unordered).inspection
@assert !unordered_map.defined && unordered_map.matrix === nothing
(; leaving_matrix_size=size(leaving_map.matrix), unordered_relation=unordered_map.relation)

## A variation: dimensions alone miss the composite

Now add a second square, $[1,3]^2$, as an independent summand. The diagonal presentation matrix says that the two summands remain independent. Along the increasing path

$$a=(1/2,1/2)\leq b=(3/2,3/2)\leq c=(5/2,5/2),$$

the spaces have dimensions $1\to2\to1$. In the overlap, the vector from the first square coexists with the vector from the second square. At the endpoint, only the second square remains.

In summand coordinates, the successive matrices are $[1\;0]^T$ and $[0\;1]$. Each has rank one; their composite is zero. The package may choose different bases, so we will check the basis-independent ranks and multiply the matrices in its consistent stored bases.

In [ ]:
two_squares = OP.encode(
    [OA.BoxUpset([0.0, 0.0]), OA.BoxUpset([1.0, 1.0])],
    [OA.BoxDownset([2.0, 2.0]), OA.BoxDownset([3.0, 3.0])],
    OP.QQ[1 0; 0 1], options)
a, b, c = (1//2, 1//2), (3//2, 3//2), (5//2, 5//2)
two_classifier = OP.encoding_map(two_squares)
two_dims = OP.dimensions(two_squares)
path_dims = [two_dims[OA.locate(two_classifier, p)] for p in (a, b, c)]
@assert path_dims == [1, 2, 1]
path_dims

## Check the maps, then read their figure

All three requests below use the same encoded module, so the target coordinates of the first matrix are the source coordinates of the second. Matrix multiplication therefore tests the composition law directly.

We display the first map, from the first square into the overlap. The printed ranks and the zero composite check also describe what happens on the rest of the path. Replace `first_spec` in the last display call with `second_spec` or `direct_spec` to inspect those maps visually.

A defined pair selection obtains a structure map and computes its rank. For a lazy encoding, obtaining the module may first compute its cover maps. An unselected inspector, a stalk selection, and `:hasse` leave lazy maps uncomputed. The inspector never requests the full table of all transitive maps.

In [ ]:
first_spec = OA.visual_spec(two_squares; kind=:module_inspector,
    parameter_pair=(a, b), box=([-1, -1], [4, 4]))
second_spec = OA.visual_spec(two_squares; kind=:module_inspector,
    parameter_pair=(b, c), box=([-1, -1], [4, 4]))
direct_spec = OA.visual_spec(two_squares; kind=:module_inspector,
    parameter_pair=(a, c), box=([-1, -1], [4, 4]))
first_map = OA.visual_metadata(first_spec).inspection
second_map = OA.visual_metadata(second_spec).inspection
direct_map = OA.visual_metadata(direct_spec).inspection
@assert (first_map.rank, second_map.rank, direct_map.rank) == (1, 1, 0)
@assert second_map.matrix * first_map.matrix == direct_map.matrix
@assert direct_map.matrix == zeros(OP.QQ, 1, 1)
display((; adjacent_ranks=(first_map.rank, second_map.rank), composite=direct_map.matrix))
display(OP.visualize(first_spec; backend=:cairomakie))

## Where did these spaces come from?

The module inspector tells us what map the finite module stores. We can now ask how the **input presentation** produced that module. A fringe presentation has an upset for each source column, a downset for each target row, and a coefficient matrix. At a parameter, keep its active rows and columns; the column space of that block is the stalk.

For our identity matrix, predict the following blocks before querying them:

| Parameter | Active rows | Active columns | Block | Dimension of its image |
| --- | --- | --- | --- | --- |
| $a$ | $D_1,D_2$ | $U_1$ | $[1\;0]^T$ | 1 |
| $b$ | $D_1,D_2$ | $U_1,U_2$ | $I_2$ | 2 |
| $c$ | $D_2$ | $U_1,U_2$ | $[0\;1]$ | 1 |

`OP.encoding_presentation` returns the finite fringe retained on this encoding's poset. This is distinct from the original geometric input. It returns `nothing` when no current finite fringe is retained; asking for inspection does not construct a new presentation. These queries use the presentation's image coordinates. They do not assume that every encoded module uses the same bases.

In [ ]:
retained_fringe = OP.encoding_presentation(two_squares)
@assert retained_fringe !== nothing
qa, qb, qc = [OA.locate(two_classifier, p) for p in (a, b, c)]
(; source=qa, overlap=qb, target=qc)

At the overlap $b$, all rows and columns are active. `presentation_stalk` returns their indices, their block, and its rank. The default `basis=false` answers this question without computing an image basis.

In [ ]:
overlap_stalk = OA.presentation_stalk(two_squares; vertex=qb)
@assert OA.active_rows(overlap_stalk) == [1, 2]
@assert OA.active_columns(overlap_stalk) == [1, 2]
@assert OA.presentation_matrix(overlap_stalk) == OP.QQ[1 0; 0 1]
@assert OA.presentation_summary(overlap_stalk).dimension == 2
OA.presentation_summary(overlap_stalk)

To see the stalk as a subspace of the active downset coordinates, request `basis=true`. Each column of `image_basis` is an actual vector in that target space. In this overlap the image is the entire two-dimensional target, so the chosen basis is the identity matrix. In the next example the distinction between the active target and its image will matter.

In [ ]:
overlap_with_basis = OA.presentation_stalk(two_squares; vertex=qb, basis=true)
@assert OA.image_basis(overlap_with_basis) == OP.QQ[1 0; 0 1]
OA.image_basis(overlap_with_basis)

The presentation inspector connects this calculation to a figure. Here we display membership in $U_2$ and $D_1$ on the actual encoding regions, then the full matrix, its active block, and the image basis. The support panels answer membership questions; a point belonging to an upset and a downset need not have a nonzero image.

`upset=2` and `downset=1` select which support panels to display. They do not alter the presentation or the active block. For inputs with no supported parameter geometry, support membership is shown on finite labels instead. All selections in this notebook are explicit API arguments.

In [ ]:
overlap_presentation_spec = OA.visual_spec(two_squares;
    kind=:presentation_inspector, point=b, basis=true,
    upset=2, downset=1, box=([-1, -1], [4, 4]))
display(OP.visualize(overlap_presentation_spec; backend=:cairomakie))

## Exercise: both supports are present, but is the image nonzero?

Move to $t=(1/2,5/2)$. This lies in $U_1$ and $D_2$, but not in $U_2$ or $D_1$. The active block keeps **row 2 and column 1** of the full identity matrix, giving the $1\times1$ matrix $[0]$. Its image has dimension zero even though the active source and target each have dimension one.

Predict the image-basis shape before running the cell: it has **one row and zero columns**, since it embeds the zero space in the one-dimensional $D_2$ target. An active zero entry and an absent row or column express different facts.

In [ ]:
t = (1//2, 5//2)
qt = OA.locate(two_classifier, t)
zero_stalk = OA.presentation_stalk(two_squares; vertex=qt, basis=true)
@assert OA.active_rows(zero_stalk) == [2]
@assert OA.active_columns(zero_stalk) == [1]
@assert OA.presentation_matrix(zero_stalk) == reshape(OP.QQ[0], 1, 1)
@assert OA.presentation_summary(zero_stalk).dimension == 0
@assert size(OA.image_basis(zero_stalk)) == (1, 0)
(; block=OA.presentation_matrix(zero_stalk),
   image_basis_size=size(OA.image_basis(zero_stalk)))

The next figure selects those two active supports. Their presence is visible, and the displayed active block remains $[0]$. The image-basis panel states its empty shape; it is not a missing result.

In [ ]:
active_zero_spec = OA.visual_spec(two_squares;
    kind=:presentation_inspector, point=t, basis=true,
    upset=1, downset=2, box=([-1, -1], [4, 4]))
display(OP.visualize(active_zero_spec; backend=:cairomakie))

## Recover a map from the embedded bases

For an ordered pair of finite labels $u\leq v$, write $B_u$ and $B_v$ for the image-basis matrices. The ambient projection $R$ drops downset coordinates that have disappeared by $v$. The induced map $C$ acts between the image coordinates and satisfies

$$B_v C = R B_u.$$

The right side projects a source image vector in downset coordinates; the left side expresses the result in the target image basis. This equation distinguishes the ambient projection from the module's structure matrix.

Along our path, $R_{a,b}=I_2$ and $R_{b,c}=[0\;1]$. The induced matrices are $C_{a,b}=[1\;0]^T$ and $C_{b,c}=[0\;1]$. `presentation_map` computes the endpoint bases needed for these maps; this work is explicit in the pair query.

In [ ]:
presentation_ab = OA.presentation_map(two_squares; source=qa, target=qb)
presentation_bc = OA.presentation_map(two_squares; source=qb, target=qc)
presentation_ac = OA.presentation_map(two_squares; source=qa, target=qc)
@assert OA.ambient_projection(presentation_ab) == OP.QQ[1 0; 0 1]
@assert OA.ambient_projection(presentation_bc) == OP.QQ[0 1]

Check the defining equation for each adjacent map. Both sides use the same active target-coordinate labels, so this is an equality of actual matrices over `QQ`.

In [ ]:
Ba = OA.image_basis(OA.source_stalk(presentation_ab))
Bb = OA.image_basis(OA.target_stalk(presentation_ab))
Bc = OA.image_basis(OA.target_stalk(presentation_bc))
@assert Bb == OA.image_basis(OA.source_stalk(presentation_bc))
@assert Bb * OA.induced_map(presentation_ab) == OA.ambient_projection(presentation_ab) * Ba
@assert Bc * OA.induced_map(presentation_bc) == OA.ambient_projection(presentation_bc) * Bb
(; source_basis=Ba, overlap_basis=Bb, target_basis=Bc)

Now multiply the induced maps. The first-square vector maps to the first coordinate in the overlap, which the second projection removes. The composite must be the $1\times1$ zero matrix.

For this diagonal presentation, the encoder and the presentation query choose the same coordinate bases, so we also compare with the matrices inspected earlier. This entrywise comparison is specific to our verified fixture. In general, two descriptions of the same module can require a change of basis; the retained presentation does not identify its image basis with an arbitrary stored module basis.

In [ ]:
@assert OA.induced_map(presentation_bc) * OA.induced_map(presentation_ab) == OA.induced_map(presentation_ac)
@assert OA.induced_map(presentation_ac) == zeros(OP.QQ, 1, 1)
@assert OA.induced_map(presentation_ab) == first_map.matrix
@assert OA.induced_map(presentation_bc) == second_map.matrix
@assert OA.induced_map(presentation_ac) == direct_map.matrix
OA.induced_map(presentation_ac)

The map view puts the source and target active blocks, their embedded bases, the ambient projection, and the induced map together. We show the step from the overlap to the second square, where the first downset coordinate disappears. Compare the projection panel with the basis panels to read the equation above.

In [ ]:
presentation_map_spec = OA.visual_spec(two_squares;
    kind=:presentation_inspector, parameter_pair=(b, c),
    upset=1, downset=2, box=([-1, -1], [4, 4]))
display(OP.visualize(presentation_map_spec; backend=:cairomakie))

## Save the inspected result

The square specification can be exported unchanged. We also save the presentation overlap, active-zero example, and induced-map view. PNG is convenient for quick sharing; SVG retains vector graphics. The canonical notebook call takes an output directory, a stem, and the specification. Its return value records the actual file path, format, backend, and recipe.

`mktempdir()` keeps this demonstration portable and avoids overwriting your files. Replace it with a persistent directory of your choice if you want the exports to remain after the Julia session closes.

In [ ]:
outdir = mktempdir()
png = OP.save_visual(outdir, "square-map", square_spec;
    format=:png, backend=:cairomakie, size=(1560, 650))
svg = OP.save_visual(outdir, "square-map", square_spec;
    format=:svg, backend=:cairomakie, size=(1560, 650))
@assert isfile(OP.export_path(png)) && isfile(OP.export_path(svg))
(; png=OP.export_path(png), svg=OP.export_path(svg))

The presentation recipes choose a layout for their additional panels. Let the specification supply its figure size when saving these views.

In [ ]:
overlap_png = OP.save_visual(outdir, "presentation-overlap", overlap_presentation_spec;
    format=:png, backend=:cairomakie)
overlap_svg = OP.save_visual(outdir, "presentation-overlap", overlap_presentation_spec;
    format=:svg, backend=:cairomakie)
zero_png = OP.save_visual(outdir, "presentation-active-zero", active_zero_spec;
    format=:png, backend=:cairomakie)
map_png = OP.save_visual(outdir, "presentation-induced-map", presentation_map_spec;
    format=:png, backend=:cairomakie)
@assert all(isfile(OP.export_path(result)) for result in (overlap_png, overlap_svg, zero_png, map_png))
(; overlap=OP.export_path(overlap_png), active_zero=OP.export_path(zero_png),
   induced_map=OP.export_path(map_png))

## What the figures retain

We followed original parameters into actual finite labels, read the corresponding spaces, and checked selected maps over rational coefficients. The two-square variation showed why dimensions and even two nonzero successive maps do not guarantee a nonzero composite. Its active presentation blocks then explained those spaces as images, and the equation $B_v C=R B_u$ explained their maps.

The active-zero point made a further distinction: an upset and a downset can both be present while their coefficient is zero. The embedded image basis records that fact as a `1 x 0` matrix. Presentation image coordinates and an arbitrary stored module's coordinates need not coincide.

These are static selections made through API arguments. Clicking a region to update another panel is not implemented yet. The graph layout is schematic, and parameter panels are clipped to the requested box; neither changes the encoding. Support colors report membership on classifier regions, whose outlines retain their boundary conventions. Distinct exact coordinates can coincide at drawing precision, which the region readout warns about.

For larger matrices, `matrix_limit=(12,12)` limits displayed rows and columns while retaining the full queried matrices. It does not cap algebraic work. A single-stalk presentation query computes an image basis only with `basis=true`; a pair query computes the bases needed for its induced map. See the [visualization guide](../visualization.md) for finite-poset-only inputs, numerical fields, selection rules, and other recipes.

We have now constructed a finite object and checked how it recovers spaces and maps from a presentation. The [tameness chapter](../tameness.md) asks when such finite descriptions exist, and distinguishes that existence question from the constructions implemented here. Interactive selection and following user-chosen vectors across panels remain later visualization work.